# How often does a Tambora happen?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&branch=main&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs/notebooks%2F07_how_often_tambora.ipynb).

In 1815 the Indonesian volcano Tambora blew its summit apart in the largest eruption anybody has
watched happen. Ash and sulphur reached the stratosphere and stayed there, and the following year
went down in Europe and eastern North America as the Year Without a Summer: frost in June, failed
harvests, bread riots. The obvious question is how often the Earth does that.

You cannot answer it by counting. There is one Tambora in the record, and one event is not a rate.
But the same catalogue holds thousands of small eruptions, California's catalogue holds thousands
of small earthquakes, and both turn out to sit on a single straight line — once you plot them the
right way. Today you will find that line where the data is thick, use it to predict something the
record has never shown you, check whether it was right, and only then take it to Tambora.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, then export the notebook as a PDF and upload that.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## What you'll be able to do

**The science.** Say how many times more small earthquakes than large ones California has, and why
that ratio is a law rather than an accident. Give a number for how often a Tambora-sized eruption
happens, and say honestly how much that number is worth.

**The skills.** Turn a column of sizes into a **cumulative count** — how many events at or above
each level. Put those counts on a **log axis** so a hopeless curve becomes a straight line. Fit
that line with `LinearRegression`, read its slope, and use `predict` to ask it about a size that is
not in your data at all.

**Eight places where you write something: five in class, three at home.** Each one is headed
*Your turn*, with an empty cell under it.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (6.5, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

def load(url, cache_name):
    """Read one live catalogue; fall back to the copy stored with the course."""
    try:
        return pd.read_csv(url)
    except Exception as e:
        print("live source unreachable, using the cached copy:", type(e).__name__)
        return pd.read_csv(CACHE + "/" + cache_name)

BOX = "&minlatitude=32&maxlatitude=42&minlongitude=-125&maxlongitude=-114"
USGS = "https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&orderby=time-asc"
GVP = ("https://webservices.volcano.si.edu/geoserver/GVP-VOTW/ows?service=WFS&version=1.0.0"
       "&request=GetFeature&typeName=GVP-VOTW:Smithsonian_VOTW_Holocene_Eruptions"
       "&outputFormat=csv")

quakes = load(USGS + "&starttime=1990-01-01&endtime=2026-01-01&minmagnitude=3.5" + BOX,
              "week07_california_1990-01-01_2026-01-01_M3.5.csv")
big_history = load(USGS + "&starttime=1810-01-01&endtime=2026-01-01&minmagnitude=7.0" + BOX,
                   "week07_california_1810-01-01_2026-01-01_M7.0.csv")
eruptions = load(GVP, "week07_gvp_eruptions.csv")

# mags is every magnitude in the box, the largest included: "how many at magnitude 5 or above"
# has to count the magnitude 7s too, or it is not that count. What we hold back today is the
# RANGE of magnitudes the line gets fitted over, never the earthquakes themselves.
mags = quakes["mag"].values
big = quakes[quakes["mag"] >= 7.0]      # the same events again, gathered so we can look at them

print("California catalogue:", quakes.shape, "  eruption catalogue:", eruptions.shape)
print("magnitude 7 and above in the same box since 1810:", len(big_history))

## Looking Tambora up in the catalogue

The Smithsonian Institution's Global Volcanism Program keeps a table of every eruption it can
document — the setup cell printed how many rows of it came back — and rates most of those on the
**Volcanic Explosivity Index**: VEI, a whole number from 0 to 8. It works like an earthquake
magnitude. From VEI 2 upwards each step stands for ten times the volume of material thrown out, so VEI 2 is a nuisance and VEI 7 rearranges the climate. (Newhall and Self defined the
index in 1982, in *Journal of Geophysical Research* **87**, 1231–1238. The tenfold step, and the
one place it breaks — from VEI 1 to VEI 2 the jump is a hundredfold, not tenfold — were read from
`en.wikipedia.org/wiki/Volcanic_explosivity_index` on 2026-08-31.)

Ask the catalogue how many eruptions of each size it holds since 1800.

In [ ]:
rated = eruptions.dropna(subset=["ExplosivityIndexMax"])
recent = rated[rated["StartDateYear"] >= 1800]

print(recent["ExplosivityIndexMax"].value_counts().sort_index())
print(recent[recent["ExplosivityIndexMax"] == 7][["Volcano_Name", "StartDateYear"]])

There it is, and there is only one of it. In the 226 years since 1800 the
catalogue holds exactly 1 eruption at VEI 7 — Tambora, whose eruptive episode the
catalogue dates from 1812, three years before the explosion that made 1816 cold.

One event gives you no rate. Divide 1 by 226 years and you get a number, but you would have
got a different number from any other stretch of history, and nothing tells you which is right.
Counting worked when we had hundreds of events to count; here it dies.

What the catalogue does have is thousands of eruptions rated VEI 2 and hundreds rated VEI 3 — read
them off the count you just printed. If the small ones and the large ones are connected by
something regular, that is thousands of measurements of the thing we want. Finding out whether they are
connected needs a catalogue where even the small events get counted properly — and the best such
catalogue on Earth is a seismic network, so we go there first.

## The shape of a catalogue of earthquakes

`quakes` is every earthquake of magnitude 3.5 and above that the USGS recorded between
1990-01-01 and 2026-01-01 inside a box around California: latitude 32 to 42 north, longitude 125 to
114 west. That box is a rectangle, not a state — it reaches into Nevada and into Baja California —
and it holds 6,287 earthquakes, whose magnitudes are all in `mags`.

Start with the plainest possible picture of them.

In [ ]:
plt.hist(mags, bins=np.arange(3.5, 8.0, 0.5))
plt.xlabel("magnitude")
plt.ylabel("number of earthquakes")
plt.title("California, 6,287 earthquakes at magnitude 3.5 and above, "
          "1990-2026")
plt.show()

A wall on the left and nothing on the right. There is no hump anywhere: almost every event sits
close to the magnitude 3.5 floor we asked for, and it would sit close to any other floor we chose,
because there are always more small ones. The events anybody actually cares about are off in a
region of the axis that looks empty.

Which is why the usual summary of a column — its middle — is worse than useless here.

In [ ]:
print("mean magnitude:  ", round(mags.mean(), 3))
print("median magnitude:", round(np.median(mags), 2))
print("largest in the box:", mags.max())

# USGS, "Earthquake Magnitude, Energy Release, and Shaking Intensity", read 2026-08-31:
# one whole step of magnitude is about 32 times more energy released.
energy_ratio = 32 ** (mags.max() - mags.mean())
# 32 is itself a rounded constant, so this answer is good to two figures and no
# more. Rounding to the nearest ten thousand is how you say that on screen.
print(f"the largest released about {round(energy_ratio, -4):,.0f} times the energy of an average one")

The average California earthquake in this file is magnitude 3.908. People nearby feel one
of those, and it damages nothing; no plan for the state's next hundred years turns on it. Meanwhile
the largest in the box released roughly 130,000 times as much energy as that
average — two figures and no more, because the 32 behind it is itself a rounded
number. When a distribution is shaped like this one, the mean describes the crowd
and the crowd is irrelevant: everything that matters is in the part of the axis where the histogram
looks like zero.

So stop asking what is typical, and start asking how the count changes as the size goes up.

### ✏️ Your turn 1

`mags` holds the magnitude of every one of the 6,287 earthquakes in the box.
Count how many are at magnitude 4.0 or above, at 5.0 or above, and at 6.0 or above — a comparison
gives you True and False, and adding those up counts the Trues. Then print each count divided by
the next one.

**Use these names**, because the self-check looks for them: `n4`, `n5`, `n6`.

In [ ]:
# ← your answer here


assert n6 < n5 < n4, "the counts must fall as the magnitude rises — check which way your >= points"
assert n4 + (mags < 4.0).sum() == len(mags), \
    ("n4 is every earthquake at 4.0 and above, so n4 and the ones below 4.0 have to account for "
     "the whole catalogue between them. If they do not, n4 is the earthquakes IN a band — a "
     "histogram bin — rather than the count at or above a level")
print("✓ counting up the catalogue —", n4, "at M4+,", n5, "at M5+,", n6, "at M6+, so",
      round(n4 / n5, 2), "and", round(n5 / n6, 2), "times fewer at each step")

One step up in magnitude and there are 11.36 times fewer earthquakes; one more step and
there are 8.14 times fewer again. Those are not the same number — they are about
40 per cent apart — and the reason to suspect one factor behind them anyway is that
the second rests on the 21 earthquakes at magnitude 6 and above. Counts of rare things wobble
by roughly their own square root, so that 21 could as easily have come out 16 or
26; at 16 the second ratio would read 10.69 against the first's
11.36. Two ratios that might be one is worth measuring properly instead of at three
points.

## Counting upwards, at every level

The histogram above counted earthquakes **in** each magnitude bin. What Your turn 1 counted is
different and more useful: how many are **at or above** a level. That is a *cumulative* count, and
it is the natural thing to ask of a hazard — nobody wants to know how many earthquakes were between
5.9 and 6.0, they want to know how many were 6 or worse.

Doing it at every level from 3.5 to 5.0 needs the list of levels first. There is a trap in building
it, so we build it in a function and use that function all week.

In [ ]:
def levels_between(lowest, highest):
    """The magnitudes lowest, lowest + 0.1, ... highest, built from whole numbers."""
    return np.arange(round(lowest * 10), round(highest * 10) + 1) / 10


mag_levels = levels_between(3.5, 5.0)
print(mag_levels)

the_obvious_way = np.arange(3.5, 5.05, 0.1)      # looks the same, and is not
print("is its fourth entry equal to 3.8?", the_obvious_way[3] == 3.8)
print("earthquakes at 3.8 and above:", (mags >= 3.8).sum(),
      "  using its fourth entry instead:", (mags >= the_obvious_way[3]).sum())

Whole numbers first, then divide. The obvious way looks identical on screen and is not: its fourth
entry is a hair above 3.8, so it excludes every earthquake recorded as exactly 3.8 and the count
falls from 3,110 to 2,896 — 214 earthquakes gone, with
no error and no warning. Decimals are stored as approximations, so two that ought to be equal often
are not. Build the levels from integers and the problem never arises.

### ✏️ Your turn 2

Write `count_at_least(values, level)`: one line, returning how many of `values` are at or above
`level`. Give it a docstring.

Then use it in a loop over `mag_levels` to build a list called `counts`, and print the first and
last entries with the level each belongs to.

**Use these names**, because the self-check looks for them: `count_at_least`, `counts`.

In [ ]:
# ← your answer here


assert len(counts) == len(mag_levels), "one count per level — is the append inside the loop?"
assert counts[0] > counts[-1], "the counts must fall as the level rises"
assert count_at_least(np.array([1.0, 2.0, 3.0]), 2.0) == 2, \
    ("'at or above' has to include the values that are exactly equal to the level — a plain > "
     "silently drops every event recorded at the level itself, and there are hundreds of those")
print("✓ cumulative counts —", len(counts), "levels, from", counts[0], "down to",
      counts[-1])

Now draw them. Level along the bottom, count up the side, one dot per level.

In [ ]:
plt.scatter(mag_levels, counts)
plt.xlabel("magnitude")
plt.ylabel("number of earthquakes at or above this magnitude")
plt.title("California, 6,287 earthquakes counted at 16 levels")
plt.show()

A curve, and one that hides the half we came for. The left-hand point is 6,287 and the
right-hand one is 171, so the axis has to reach 6,287 and the
8 dots of the right half are all squeezed into the bottom 15
per cent of it. You can see that they descend. What you cannot see is by how much: reading the
factor between magnitude 4.5 and magnitude 5.0 off that band means comparing two dots sitting at
9 and 3 per cent of the axis height — and that factor is the whole
question.

The fix is the one the plotting week introduced. When the values span factors of a thousand, plot
the exponents instead and a curve becomes a line. `plt.yscale("log")` does exactly that: the
distance from 10 to 100 on the axis becomes the same as the distance from 100 to 1000.

In [ ]:
plt.scatter(mag_levels, counts)
plt.yscale("log")
plt.xlabel("magnitude")
plt.ylabel("number of earthquakes at or above this magnitude")
plt.title("California, the same 16 counts of 6,287 earthquakes, log axis")
plt.show()

The same numbers, and now they are a straight line.

A straight line on a log count axis has a name and a meaning. Every step up in size divides the
count by the same factor — which lets you predict the sizes you have never seen. Seismologists call
this the Gutenberg–Richter relation, after the two Caltech seismologists who described it in the
1940s, and it holds with the same shape in essentially every region anybody has looked at.

## Measuring the line

A straight line is what least squares is for. The log axis was a change of drawing and not of
data, so the counts themselves are still a curve; what is straight is `np.log10(counts)` plotted
against magnitude, and that is the pair least squares gets. Draw the best straight line. Best means
the smallest total miss.

`LinearRegression` wants one row per data point rather than one long row, which is what
`.reshape(-1, 1)` is doing below: sixteen numbers become sixteen rows of one number each.

### ✏️ Your turn 3

Fit a straight line through `mag_levels` and `np.log10(counts)`:

```
model = LinearRegression()
model.fit(mag_levels.reshape(-1, 1), np.log10(counts))
```

Then print `model.coef_[0]` as `slope`, print `model.intercept_`, print the R-squared from
`model.score(...)` on the same two arguments, and finally print `10 ** -slope` — which is how many
times fewer earthquakes there are for each whole step up in magnitude.

**Use these names**, because the self-check looks for them: `model`, `slope`.

In [ ]:
# ← your answer here


assert -1.5 < slope < -0.5, ("a slope near -1 is expected here; if yours is far from that, "
                             "check that you fitted np.log10(counts) and not counts")
print("✓ the fitted line — slope", round(slope, 3), "so each whole magnitude step "
      "divides the count by", round(10 ** -slope, 2))

A slope of -1.037, with R squared 0.99976. Seismologists quote the size of that slope and
call it the **b-value**; a b-value near 1 is what Gutenberg and Richter found and what most of the
world's crust gives. It says that earthquakes of magnitude 4 and above outnumber those of
magnitude 5 and above by about 10.9 to one, and that the same factor holds between
any two levels a whole magnitude apart, with no favoured size anywhere.

That is a strong statement about how the crust breaks. Rock does not have a characteristic
earthquake size the way a person has a characteristic height — the same physics of a rupture
running along a fault and stopping produces every size, and only the chance of running further
decides which one you get.

The one number in that output you should not be impressed by is the R squared. A cumulative count
can only fall as the level rises — that is what "at or above" means — so these dots were going to
descend smoothly whatever the magnitudes were, and a fit to something already smooth and already
falling scores near 1 almost regardless. Test it: give the identical fit magnitudes with no pattern
in them at all, spread evenly from one end of the catalogue's range to the other.

In [ ]:
# the same number of magnitudes, with no law in them: every tenth from 3.5 to
# 7.3 equally likely, drawn in tenths so they land on the same grid as real magnitudes
flat = np.random.default_rng(88).integers(35, 74, len(mags)) / 10

flat_counts = []
for level in mag_levels:
    flat_counts.append(count_at_least(flat, level))

flat_model = LinearRegression()
flat_model.fit(mag_levels.reshape(-1, 1), np.log10(flat_counts))
print("R squared on magnitudes with no pattern at all:",
      round(flat_model.score(mag_levels.reshape(-1, 1), np.log10(flat_counts)), 5))

0.99686 — from magnitudes that hold no law whatever, against 0.99976 from the real
catalogue. The check could hardly have failed, so passing it says almost nothing, and any argument
resting on that 0.99976 is resting on the shape of a cumulative count rather than on California.
What would be worth something is the line holding up somewhere it was never fitted. That is
testable, so test it.

## Reading the line off past the end of the data

The line was measured between magnitude 3.5 and 5.0. Nothing stops us evaluating it somewhere else,
and `model.predict` will do it without complaint — so the next cell draws the line right across the
plot, well past the last dot it was fitted to.

In [ ]:
line_x = levels_between(3.5, 7.5)
line_y = 10 ** model.predict(line_x.reshape(-1, 1))

plt.scatter(mag_levels, counts, label="counted")
plt.plot(line_x, line_y, color="firebrick", label="the fitted line, extended")
plt.yscale("log")
plt.xlabel("magnitude")
plt.ylabel("number of earthquakes at or above this magnitude")
plt.title("California, 6,287 earthquakes and the line fitted to 16 levels")
plt.legend()
plt.show()

In [ ]:
predicted_7 = 10 ** model.predict([[7.0]])[0]
print("the line expects", round(predicted_7, 2), "earthquakes at magnitude 7 or above")

### Predict before you run

The line, which was measured between magnitude 3.5 and 5.0 and never asked about anything larger,
says to expect about 1.49 earthquakes of magnitude 7 or above in this box in these
36 years.

How many actually happened? Write your guess into `my_guess` below before you run the cell. `big`
holds them, gathered in the setup cell and not looked at since.

In [ ]:
my_guess = 2

print("you guessed:", my_guess)
print("the catalogue holds:", len(big))
print(big[["time", "mag", "place"]].to_string(index=False))

5 of them — about 3.4 times what the line expected. They are real and they
are famous: two in 1992, then 1999, 2010, 2019. Notice also that one of the five is in
Baja California, Mexico: the box is a rectangle drawn on a map, and rectangles do not respect
borders. Whatever this section concludes is about that rectangle, not about the state.

Before blaming the line, check whether the line was ever a single thing. We chose to fit it between
magnitude 3.5 and 5.0. Somebody else would have chosen differently, and the honest question is
whether that choice is doing the work.

### ✏️ Your turn 4

Write `predict_count(values, levels, target)`, which packages up what you did in Your turn 2 and
Your turn 3:

- build the cumulative count of `values` at each level in `levels`, using your `count_at_least`
- fit a `LinearRegression` to `levels.reshape(-1, 1)` and `np.log10` of those counts
- **return two things**: the slope, and `10 ** model.predict([[target]])[0]`

Give it a docstring. Then loop over the three fitting ranges `(3.5, 5.0)`, `(4.0, 5.5)` and
`(4.5, 6.0)`, building the levels for each with `levels_between`, and print the slope and the
predicted number at magnitude 7 for each. Collect the three predictions in a list called
`predictions`.

The first range is the one you already did by hand, so its answer is a check on your function.

**Use these names**, because the self-check looks for them: `predict_count`, `predictions`.

In [ ]:
# ← your answer here


assert len(predictions) == 3, "three fitting ranges, three predictions"
assert min(predictions) > 0.5, ("a prediction below 0.5 usually means the 10 ** was left off, so "
                                "the function is returning the log of the count")
assert max(predictions) - min(predictions) > 0.1, \
    ("three different fitting ranges cannot give three identical answers — build the levels "
     "afresh inside the loop, one set per range, rather than passing the same ones each time")
print("✓ three defensible choices — the line expects",
      round(min(predictions), 2), "to", round(max(predictions), 2),
      "at magnitude 7, against", len(big), "that happened")

The three defensible ranges land between 1.49 and
2.21, and 5 happened. The choice does matter: the highest of the
three expects about 1.5 times what the
lowest does, which is worth remembering the next time somebody quotes a single number off a fit like
this one. What it does not do is close the gap. Every range falls short of 5 by a factor
of between 2.3 and
3.4, so whatever is missing is missing from all three.

Three things could be true, and this notebook cannot tell them apart. 36 years may simply be
too short a window for an event this rare, so we are looking at an unlucky draw. Or California may
genuinely make more large earthquakes than its small ones imply: the small events come from cracked
crust everywhere, while the magnitude 7s come from a handful of very long faults rupturing along
most of their length, and whether those long faults deliver more big earthquakes than the
small-event line allows is a live argument in seismology rather than a settled question. Or the
counting could be off. Hold the question open; the first part of the homework goes after the first
of those three.

## The same line, drawn for volcanoes

VEI is a magnitude scale, so the same machinery applies without changing a thing: cumulative counts
at each level, log axis, straight line, and then read the line off at VEI 7 where the catalogue has
almost nothing.

The counts below are every rated eruption since 1800, at each level from 0 upward.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
# Re-run your own count_at_least (Your turn 2) and predict_count (Your turn 4) cells as well.
# Those two are your code, so this cell cannot rebuild them for you; the rest of the week uses them.
rated = eruptions.dropna(subset=["ExplosivityIndexMax"])
recent = rated[rated["StartDateYear"] >= 1800]
vei = recent["ExplosivityIndexMax"].values

In [ ]:
counts_by_vei = []
for level in range(0, 7):
    counts_by_vei.append(count_at_least(vei, level))
    print("VEI", level, "and above:", counts_by_vei[-1])

print()
for level in range(1, 7):
    print("from VEI", level - 1, "to VEI", level, ": divided by",
          round(counts_by_vei[level - 1] / counts_by_vei[level], 1))

Read the second block of numbers. From VEI 2 upwards each step divides the count by roughly the
same factor, somewhere near five every time — but the bottom two steps do not play along at all.
Going from VEI 2 down to VEI 1 the count should multiply by about five, and it barely moves.

That is not volcanology, and it is not the index either. The one place VEI departs from the tenfold
rule is the step from VEI 1 to VEI 2, and it departs the wrong way for this: a VEI 1 covers a
hundredfold range of volumes rather than a tenfold one, so there should be **more** VEI 1 eruptions
than the line asks for, not fewer. What is left is bookkeeping. A catalogue lists what somebody's
instruments recorded, not what happened. Where there are no seismometers there are no earthquakes in
the file — and nobody files a report on a VEI 1 eruption in an uninhabited part of the Andes in
1840. So the fit starts at VEI 2, where the record is thick enough to trust.

### ✏️ Your turn 5

Fit the line to VEI 2, 3 and 4 only, and ask it about VEI 7.

Build `vei_levels = np.array([2, 3, 4])`, call your `predict_count` on `vei` with `target` 7, and
print the slope, the predicted number of VEI 7 eruptions since 1800, and — using
`count_at_least` — how many the catalogue actually holds.

**Use these names**, because the self-check and the cells below look for them: `vei_levels`,
`vei_slope`, `vei_predicted`.

In [ ]:
# ← your answer here


assert not isinstance(vei_predicted, tuple), \
    "predict_count hands back two things — unpack both: vei_slope, vei_predicted = predict_count(...)"
assert 0.5 < vei_predicted < 2, \
    ("a VEI 7 count outside 0.5 to 2 means something went into predict_count in the wrong slot — "
     "the order is the values, then the levels, then the target")
print("✓ the volcano line — it expects", round(vei_predicted, 2),
      "at VEI 7 and the catalogue holds", count_at_least(vei, 7))

In [ ]:
all_levels = np.arange(0, 8)

vei_counted = []
vei_on_the_line = []
for level in all_levels:
    vei_counted.append(count_at_least(vei, level))
    vei_on_the_line.append(predict_count(vei, vei_levels, level)[1])
    print("VEI", level, " counted", vei_counted[-1],
          "  the line says", round(vei_on_the_line[-1], 2))

In [ ]:
plt.scatter(all_levels, vei_counted, label="counted")
plt.plot(all_levels, vei_on_the_line, color="firebrick", label="the line fitted to VEI 2, 3, 4")
plt.yscale("log")
plt.xlabel("Volcanic Explosivity Index")
plt.ylabel("number of eruptions at or above this VEI")
plt.title(f"Eruptions since 1800, {len(vei):,} of them rated")
plt.legend()
plt.show()

Above VEI 4 the line does something it had no obligation to do: it lands close to counts it was
never shown. Compare the two columns you just printed at VEI 5, VEI 6 and VEI 7.

Only the VEI 5 row carries much weight. There are enough eruptions there that the line had room to
miss and did not. The two rows above it hold a handful and one, and what agreement on a handful is
worth is the next thing we take up.

Below VEI 2 the line is a disaster, and the log axis is what makes the size of the disaster visible:
at VEI 0 and above it calls for close to twenty times what the catalogue holds — the top row of the
same table. Taken literally that says more than nine in ten of the world's smallest
eruptions never reached anybody's records; taken carefully it says the record cannot be trusted
down there at all, which is the same conclusion and a safer way to say it. Either way the missing
thing is the data, not the line.

## What one observation can settle

About one predicted against exactly 1 observed looks like a triumph, and this is
exactly the moment to be suspicious. A prediction is only impressive if it could have been
embarrassed, so ask what would have counted as a failure here: if the true rate of VEI 7 eruptions
were something quite different, how often would 226 years still hand you exactly one?

Rare independent events arrive with Poisson counts, so simulate. Twenty thousand imaginary
histories at each of several rates, counting how often each one produces exactly 1.

In [ ]:
rng = np.random.default_rng(88)

for rate in [0.1, 0.5, 1.0, 2.0, 4.0]:
    histories = rng.poisson(rate, size=20000)
    print("if the true rate were", rate, "per", 226, "years, exactly 1 happens in",
          round((histories == 1).mean() * 100), "% of histories")

# GVP publishes the Holocene record, and the Holocene began about 9,700 BCE.
# Anchor the window on the record itself; anchoring it on the oldest eruption IN the record
# shortens the span and overstates the rate.
record_start = -9700
record_span = 2026 - record_start
holocene = rated[rated["StartDateYear"] >= record_start]
top = holocene[holocene["ExplosivityIndexMax"] >= 7]

print()
print("the fitted rate is one VEI 7 every", round(226 / vei_predicted), "years")
print("the whole record holds", len(top), "of them in", record_span,
      "years — one every", round(record_span / len(top)), "years")

Every rate from 0.1 to 4.0 produces exactly one eruption in a
respectable fraction of histories — between 8 and
37 per cent. A single count cannot tell those apart, so the fitted
rate and a rate of 4.0 both "pass" this test, and so would a
model wrong by a factor of 40. The check has almost no
power to fail, which means passing it carries almost no information.

And the wider record says the same thing from the other direction. Over the whole Holocene —
9,700 BCE to now — 7 eruptions reached VEI 7, one every
1,675 years, around 7 times rarer than the
post-1800 fit says. The old record is certainly missing some, so the truth is somewhere
between. Notice which case is which: the earthquake prediction failed loudly and the volcano
prediction passed quietly, and it is the loud failure that told us something.

## The question, answered

**Roughly once every few hundred to a couple of thousand years, and this week's data cannot do
better than that.** Fitting the Gutenberg–Richter line to eruptions of VEI 2 to 4 since
1800 predicts about one eruption at VEI 7 in that 226-year window — a rate of one
every couple of centuries — and exactly 1 occurred. Across the whole Holocene record
the same kind of event has come about once every 1,675 years. Both numbers are
defensible and they disagree by a factor of 7, because one rests on a single event
and the other on a record that thins as it goes back.

The line itself is the solid part, and one check is carrying that: fitted to VEI 2, 3 and 4, it
reproduces the count at VEI 5 without ever being shown it, and there are enough eruptions at VEI 5 —
a couple of dozen — that it had room to miss. That is the load-bearing test, and it is the only one.
Landing near the count at VEI 6 is not a second one: a handful of eruptions is the same near-empty
situation as the 1 at VEI 7, where every rate from 0.1 to
4.0 passed. Neither is the R squared — magnitudes with no pattern in them scored
0.99686 on the same fit. Where the line was tested by something that could have refuted it,
it duly was refuted: California's magnitude 7s outnumber what it expects by a factor of
3.4.

## Week 7 summary

**The question.** How often does a Tambora happen?

### What to remember

| | |
|---|---|
| **1** | Small earthquakes vastly outnumber large ones, and the ratio is a law rather than an accident. |
| **2** | A straight line on log axes lets you predict events too rare to have been observed at all. |
| **3** | A prediction that matches a single observation is not evidence — one count cannot falsify anything. |

### The ideas, in plain words

| Idea | Means |
|---|---|
| **Log axes** | When the values span factors of a thousand, plot the exponents instead and a curve becomes a line. |
| **Power law** | Every step up in size divides the count by the same factor — which lets you predict the sizes you have never seen. |

### Code you met this week

| Function | What it does |
|---|---|
| `np.log10(values)` | the exponent of every value at once — what turns a power law into a straight line |
| `array.reshape(-1, 1)` | one row per data point, which is the shape scikit-learn wants |

## Homework

Three parts, on the same two catalogues. Part 1 goes after the first of the three explanations class
left open for California; part 2 makes you choose a window for the volcanoes and live with the
consequence; part 3 puts the two together. If you have restarted since class, run the setup cell at
the top first, then the checkpoint below.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
# Re-run your own count_at_least (Your turn 2) and predict_count (Your turn 4) cells as well.
# Those two are your code, so this cell cannot rebuild them for you; every part below uses them.
rated = eruptions.dropna(subset=["ExplosivityIndexMax"])
vei_levels = np.array([2, 3, 4])

### ✏️ Your turn 6

Class found 5 earthquakes of magnitude 7 or above in 36 years, against a line that
expected 1.49. The first explanation on the list was that 36 years is too short a
window and we drew an unlucky one. That is checkable: count the other windows.

`big_history` holds every magnitude 7 and above the catalogue has for the same box back to
1810 — 15 of them. Get the year of each with
`big_history["time"].str[:4].astype(int)`, then loop over
`window_starts = range(1810, 2026, 36)` and for each one count the events with
year at or after `start` and before `start + 36`. Collect the counts in a list called
`window_counts`, print each window with its count, and print the mean.

**Use these names**, because the self-check looks for them: `window_counts`, `big_history`.

In [ ]:
# ← your answer here


assert sum(window_counts) == len(big_history), \
    "every event should land in exactly one window — check for < rather than <= at the top edge"
print("✓ six windows — counts", np.array(window_counts), "with a mean of",
      round(np.mean(window_counts), 2))

### ✏️ Your turn 7

Class fitted the volcano line to eruptions since 1800. That start year was a choice, and
two others are just as defensible: 1700, which buys more eruptions at the cost of a
thinner record, and 1900, which buys a better record at the cost of fewer eruptions.

For each of `[1700, 1900]`: cut `rated` down to eruptions with `StartDateYear` at or after that
year, call `predict_count` on their `ExplosivityIndexMax` values with `vei_levels` and target 7, and
print the start year, how many eruptions the window holds, the predicted number at VEI 7, and — with
`count_at_least` — how many actually occurred in that window. Collect the predictions in a list
called `fork_predictions`.

**Use these names**, because the self-check looks for them: `fork_predictions`, `rated`,
`vei_levels`.

In [ ]:
# ← your answer here


assert len(fork_predictions) == 2, "two start years, two predictions"
assert fork_predictions[0] != fork_predictions[1], \
    "identical predictions mean the same eruptions went in twice — check the cut inside the loop"
print("✓ the window is a choice — the two start years predict",
      round(fork_predictions[0], 2), "and", round(fork_predictions[1], 2), "VEI 7 eruptions")

### ✏️ Your turn 8

Two or three sentences, using your own numbers from parts 1 and 2, on this question: **does anything
you computed rescue the California line, and does anything you computed threaten the volcano line?**

Quote the six window counts and their mean against the 1.49 the line expected, and quote
both of your part 2 predictions against the counts that went with them. Do not answer from the
summary table; answer from your output.

*(Double-click this cell and replace this line with your answer.)*